# This notebook will show you how to pull data using a list of ICD10 codes 

In [ ]:
import pandas as pd
#codes = pd.read_csv('/home/jupyter/workspace/workspace-bucket/kristin_files/data/icd10_codes/all_field_ids_UKB_icd10_codes_aug_05_25.csv')
#codes = codes[codes['Cohort']=='UKB'] # grab UKB
#codes = codes[codes['Type'] == 'virus']

codes = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/bacterial_codes1.csv')
codes.head()

In [ ]:
code_list = list(codes['ICD10'])
# Flatten and remove duplicates
flattened = set(code.strip() for entry in code_list for code in entry.split(','))
icd10_prefixes = sorted(flattened)
icd10_prefixes = [item for item in icd10_prefixes]
print(len(icd10_prefixes))
icd10_prefixes

In [ ]:
#enter the codes you want here
print(icd10_prefixes)

In [ ]:
import pandas
import os

#Run this code to pull the above prefixes
dataset_47325552_condition_sql = f"""
SELECT
    co.person_id,
    co.condition_concept_id,
    c_standard.concept_name AS standard_concept_name,
    c_standard.concept_code AS standard_concept_code,
    c_standard.vocabulary_id AS standard_vocabulary,
    co.condition_start_datetime,
    co.condition_end_datetime,
    co.condition_type_concept_id,
    c_type.concept_name AS condition_type_concept_name,
    co.stop_reason,
    co.visit_occurrence_id,
    visit.concept_name AS visit_occurrence_concept_name,
    co.condition_source_value,
    co.condition_source_concept_id,
    c_source.concept_name AS source_concept_name,
    c_source.concept_code AS source_concept_code,
    c_source.vocabulary_id AS source_vocabulary,
    co.condition_status_source_value,
    co.condition_status_concept_id,
    c_status.concept_name AS condition_status_concept_name
FROM
    `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` co
LEFT JOIN
    `{os.environ["WORKSPACE_CDR"]}.concept` c_standard
    ON co.condition_concept_id = c_standard.concept_id
LEFT JOIN
    `{os.environ["WORKSPACE_CDR"]}.concept` c_type
    ON co.condition_type_concept_id = c_type.concept_id
LEFT JOIN
    `{os.environ["WORKSPACE_CDR"]}.visit_occurrence` v
    ON co.visit_occurrence_id = v.visit_occurrence_id
LEFT JOIN
    `{os.environ["WORKSPACE_CDR"]}.concept` visit
    ON v.visit_concept_id = visit.concept_id
LEFT JOIN
    `{os.environ["WORKSPACE_CDR"]}.concept` c_source
    ON co.condition_source_concept_id = c_source.concept_id
LEFT JOIN
    `{os.environ["WORKSPACE_CDR"]}.concept` c_status
    ON co.condition_status_concept_id = c_status.concept_id
WHERE
    c_source.vocabulary_id = 'ICD10CM'
    AND (
        {" OR ".join([f"c_source.concept_code LIKE '{p}%'" for p in icd10_prefixes])}
    )
"""


In [ ]:
dataset_47325552_condition_df = pandas.read_gbq(
    dataset_47325552_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

dataset_47325552_condition_df.head()


# Check to see which cutoff date is best to use

In [ ]:
# Rename df
df = dataset_47325552_condition_df
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert to datetime
df['condition_start_datetime'] = pd.to_datetime(df['condition_start_datetime'], format='mixed')

# Count prescriptions by year
year_counts = (
    df['condition_start_datetime']
    .dt.year
    .value_counts()
    .sort_index()
)

# Calculate threshold
avg_count = year_counts.mean()
threshold = avg_count * 0.25

print(f"Average count: {avg_count:.1f}")
print(f"25% threshold: {threshold:.1f}")

# Find first year below threshold
sparse_years = year_counts[year_counts < threshold]

if len(sparse_years) > 0:
    cutoff_year = sparse_years.index.max() + 1
    print(f"Suggested cutoff year: {cutoff_year}")

In [ ]:
plt.figure(figsize=(10,5))
year_counts.plot(kind='bar')

plt.axhline(
    threshold,
    linestyle='--',
    label=f'25% of average ({threshold:.0f})'
)

plt.ylabel('Number of ICD10 records')
plt.xlabel('Year')
plt.title('ICD10 records by year')
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert to datetime
df['condition_start_datetime'] = pd.to_datetime(df['condition_start_datetime'], format='mixed')

# Count prescriptions by year
year_counts = (
    df['condition_start_datetime']
    .dt.year
    .value_counts()
    .sort_index()
)

# Calculate threshold
recent_avg = year_counts[year_counts.index >= 2010].mean()
threshold = recent_avg * 0.25

print(f"Average count: {recent_avg:.1f}")
print(f"25% threshold: {threshold:.1f}")

# Find first year below threshold
sparse_years = year_counts[year_counts < threshold]

if len(sparse_years) > 0:
    cutoff_year = sparse_years.index.max() + 1
    print(f"Suggested cutoff year: {cutoff_year}")

In [ ]:
plt.figure(figsize=(10,5))
year_counts.plot(kind='bar')

plt.axhline(
    threshold,
    linestyle='--',
    label=f'25% of average ({threshold:.0f})'
)

plt.ylabel('Number of medication records')
plt.xlabel('Year')
plt.title('Exposure records by year')
plt.legend()
plt.show()

In [ ]:
df.to_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/all_bacterialcodes_data.csv', header = True, index = False)

In [ ]:
df.head()

In [ ]:
import pandas as pd
chunks = []

for chunk in pd.read_csv(
    '/home/jupyter/workspace/WORKSPACE_BUCKET/data/all_bacterialcodes_data.csv',
    usecols=['person_id', 'standard_concept_name', 'condition_start_datetime', 'condition_source_value'],
    chunksize=1_000_000
):
    chunk['start_date'] = pd.to_datetime(
        chunk['condition_start_datetime'].str[:10],
        errors='coerce'
    )

    chunk = chunk.loc[
        chunk['start_date'] >= '2015-01-01',
        ['person_id', 'standard_concept_name', 'start_date', 'condition_source_value']
    ]

    chunks.append(chunk)

test = pd.concat(chunks, ignore_index=True)
test = test.sort_values('start_date')


In [ ]:
test

In [ ]:
df = test

In [ ]:
# Look at concept name value counts
df.standard_concept_name.value_counts()


In [ ]:
# Split ICD10 code to get groups like in UKB
df['icd10_group'] = df.condition_source_value.str.split('.').str[0]
df

In [ ]:
# Create a list of ICD10 codes
code_list = list(set(list(df['icd10_group'])))
print(len(code_list))
sorted_list = sorted(code_list)
print(sorted_list)

In [ ]:
# shows the number of participants for each icd_10 code
for code in sorted_list:
    test = df[df['icd10_group']==code]
    test = test[['person_id', 'standard_concept_name', 'start_date', 'icd10_group']]
    test = test.sort_values(by = 'start_date')
    test = test.drop_duplicates(subset = 'person_id', keep = 'first')
    print(code, len(test))
    test.to_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/ICD10_Codes/{code}_with_date_2015.csv', header = True, index = False)

In [ ]:
import pandas as pd

test_code = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/ICD10_Codes/B95_with_date_2015.csv')
test_code

In [ ]:
# Check one phenocode
test = codes[codes['endpoint_code']=='K11_ORAL']
test = test[['endpoint_code', 'endpoint_name', 'ICD10']]
test

In [ ]:
condition_list = list(codes['endpoint_code'])
print(condition_list)
print(len(condition_list))

In [ ]:

for condition in condition_list:
    print(condition)
    df = pd.DataFrame()
    c2 = codes[codes['endpoint_code'] == condition]
    code_list = list(c2['ICD10'])
    print(code_list)

   # Flatten and remove duplicates
    flattened = set(code.strip() for entry in code_list for code in entry.split(','))
    #remove = ['J13', 'J14', 'J15', 'J16', 'J17', 'A48', 'A49']
    remove=[]
    
    # Convert back to a sorted list if desired
    unique_codes = sorted(flattened)
    unique_codes = [item for item in unique_codes if item not in remove]
    print(unique_codes)

    for code in unique_codes:
        test = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/ICD10_Codes/{code}_with_date_2015.csv')
        test = test.rename(columns = {'person_id':'ID', 'start_date':condition, 'icd10_group':'code'})
        df = pd.concat([df, test])
        
    #remove duplicate IDs, keeping the first condition
    df = df.sort_values(by = condition)
    df = df.drop_duplicates(subset = 'ID', keep = 'first')
    df.to_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/ICD10_Codes/Finngen_codes/{condition}.csv', header = True, index = False)